# PYPOWER DC-OPF on the PJM 5-Bus Network (Reference Solution)
DC optimal power flow on the PJM 5-bus case (pglib case5_pjm) solved with PYPOWER. This notebook is the reference solution for the Pyomo DC-OPF notebook: it provides an independent dispatch, cost, bus LMPs and line shadow prices to compare against.

**Contents**
* Setup
* Case Data
* Run DC-OPF
* Results
* Reference Values for the Pyomo Notebook


## Setup
PYPOWER is a Python port of MATPOWER. `dcopf` builds the DC optimal power flow and solves it with PIPS, an interior-point solver. Install once with `pip install pypower`.

In [1]:
!pip install pypower -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.6/347.6 kB 5.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from pypower.api import dcopf, ppoption
from pypower.idx_bus import BUS_I, VA, LAM_P            # bus: id, voltage angle (deg), LMP ($/MWh)
from pypower.idx_gen import GEN_BUS, PG                 # gen: bus, active output (MW)
from pypower.idx_brch import F_BUS, T_BUS, RATE_A, PF, MU_SF, MU_ST   # branch: ends, limit, flow, multipliers

pd.options.display.float_format = "{:,.4f}".format

## Case Data
Same network as the Pyomo notebook: 5 buses, 5 generators and 6 lines, with Branch 6 (Bus 4 - Bus 5, 240 MW) as the tightest line. PYPOWER takes the case as a dictionary of arrays in MATPOWER format. Power is in MW, and `gencost` uses the polynomial model (`2`) with 3 coefficients `c2, c1, c0`. With `c2 = 0` every cost is linear in \$/MWh.

In [3]:
net = {
    "baseMVA": 100.0,
    
    # areas: area, reference bus
    "areas": np.array([[1, 4]]),

    # bus: bus_i, type, Pd, Qd, Gs, Bs, area, Vm, Va, baseKV, zone, Vmax, Vmin
    "bus": np.array([
            [1, 2, 0.0, 0.0, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [2, 1, 300.0, 98.61, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [3, 2, 300.0, 98.61, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [4, 3, 400.0, 131.47, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [5, 2, 0.0, 0.0, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000]
        ]),

    # gen: bus, Pg, Qg, Qmax, Qmin, Vg, mBase, status, Pmax, Pmin
    "gen": np.array([
            [1, 20.0, 0.0, 30.0, -30.0, 1.0, 100.0, 1, 40.0, 0.0],
            [1, 85.0, 0.0, 127.5, -127.5, 1.0, 100.0, 1, 170.0, 0.0],
            [3, 260.0, 0.0, 390.0, -390.0, 1.0, 100.0, 1, 520.0, 0.0],
            [4, 100.0, 0.0, 150.0, -150.0, 1.0, 100.0, 1, 200.0, 0.0],
            [5, 300.0, 0.0, 450.0, -450.0, 1.0, 100.0, 1, 600.0, 0.0]
        ]),

    # gencost: model (2 = polynomial), startup, shutdown, n coefficients, c2, c1, c0
    "gencost": np.array([
            [2, 0.0, 0.0, 3, 0.000000, 14.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 15.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 30.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 40.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 10.000000, 0.000000],
    ]),

    # branch: fbus, tbus, r, x, b, rateA, rateB, rateC, ratio, angle, status, angmin, angmax    
    "branch": np.array([
            [1, 2, 0.00281, 0.0281, 0.00712, 400.0, 400.0, 400.0, 0.0, 0.0, 1, -30.0, 30.0],
            [1, 4, 0.00304, 0.0304, 0.00658, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [1, 5, 0.00064, 0.0064, 0.03126, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [2, 3, 0.00108, 0.0108, 0.01852, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [3, 4, 0.00297, 0.0297, 0.00674, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [4, 5, 0.00297, 0.0297, 0.00674, 240.0, 240.0, 240.0, 0.0, 0.0, 1, -30.0, 30.0]
        ])
}



In [4]:
# the same data as labelled tables, for reading
bus_cols = ["bus_i", "type", "Pd", "Qd", "Gs", "Bs", "area", "Vm", "Va", "baseKV", "zone", "Vmax", "Vmin"]
gen_cols = ["bus", "Pg", "Qg", "Qmax", "Qmin", "Vg", "mBase", "status", "Pmax", "Pmin"]
branch_cols = ["fbus", "tbus", "r", "x", "b", "rateA", "rateB", "rateC", "ratio", "angle", "status", "angmin", "angmax"]
cost_cols = ["model", "startup", "shutdown", "n", "c2", "c1", "c0"]

with pd.option_context("display.float_format", "{:g}".format):      # plain numbers (1, not 1.0000)
    print("Bus data");       display(pd.DataFrame(net["bus"], columns=bus_cols))
    print("Generator data"); display(pd.DataFrame(net["gen"], columns=gen_cols))
    print("Branch data");    display(pd.DataFrame(net["branch"], columns=branch_cols))
    print("Generator cost data"); display(pd.DataFrame(net["gencost"], columns=cost_cols))

Bus data


,bus_i,type,Pd,Qd,Gs,Bs,area,Vm,Va,baseKV,zone,Vmax,Vmin
0,1,2,0,0,0,0,1,1,0,230,1,1.1,0.9
1,2,1,300,98.61,0,0,1,1,0,230,1,1.1,0.9
2,3,2,300,98.61,0,0,1,1,0,230,1,1.1,0.9
3,4,3,400,131.47,0,0,1,1,0,230,1,1.1,0.9
4,5,2,0,0,0,0,1,1,0,230,1,1.1,0.9


Generator data


,bus,Pg,Qg,Qmax,Qmin,Vg,mBase,status,Pmax,Pmin
0,1,20,0,30,-30,1,100,1,40,0
1,1,85,0,127.5,-127.5,1,100,1,170,0
2,3,260,0,390,-390,1,100,1,520,0
3,4,100,0,150,-150,1,100,1,200,0
4,5,300,0,450,-450,1,100,1,600,0


Branch data


,fbus,tbus,r,x,b,rateA,rateB,rateC,ratio,angle,status,angmin,angmax
0,1,2,0.00281,0.0281,0.00712,400,400,400,0,0,1,-30,30
1,1,4,0.00304,0.0304,0.00658,426,426,426,0,0,1,-30,30
2,1,5,0.00064,0.0064,0.03126,426,426,426,0,0,1,-30,30
3,2,3,0.00108,0.0108,0.01852,426,426,426,0,0,1,-30,30
4,3,4,0.00297,0.0297,0.00674,426,426,426,0,0,1,-30,30
5,4,5,0.00297,0.0297,0.00674,240,240,240,0,0,1,-30,30


Generator cost data


,model,startup,shutdown,n,c2,c1,c0
0,2,0,0,3,0,14,0
1,2,0,0,3,0,15,0
2,2,0,0,3,0,30,0
3,2,0,0,3,0,40,0
4,2,0,0,3,0,10,0


## Run DC-OPF
`dcopf` returns the case dictionary with the solution filled in (generator output, bus angles, line flows and the dual variables), plus `success` and the objective value `f`. `PF_DC=1` selects the DC formulation.

In [5]:
option = ppoption(PF_DC=1)
results = dcopf(net, option)

print(f"Converged: {results['success']}")
print(f"Total cost: ${results['f']:,.2f}/h")

PYPOWER Version 5.1.21, 01-Aug-2026 -- DC Optimal Power Flow
Python Interior Point Solver - PIPS, Version 1.0, 07-Feb-2011
Converged!
Converged: True
Total cost: $17,479.90/h


## Results

### 1. Generator dispatch

In [6]:
gen_df = pd.DataFrame({
    "Gen": np.arange(1, len(results["gen"]) + 1),
    "Bus": results["gen"][:, GEN_BUS].astype(int),
    "Cost ($/MWh)": results["gencost"][:, 5],                      # c1 (linear cost)
    "Pg (MW)": results["gen"][:, PG],
})
gen_df["Operating cost ($/h)"] = gen_df["Pg (MW)"] * gen_df["Cost ($/MWh)"]
gen_df

,Gen,Bus,Cost ($/MWh),Pg (MW),Operating cost ($/h)
0,1,1,14.0000,40.0000,560.0000
1,2,1,15.0000,170.0000,"2,550.0000"
2,3,3,30.0000,323.4948,"9,704.8454"
3,4,4,40.0000,0.0000,0.0000
4,5,5,10.0000,466.5052,"4,665.0515"


Gen 5 (cheapest) is held below its 600 MW maximum and Gen 3 covers the difference.

### 2. Bus angles and LMPs

In [7]:
bus_df = pd.DataFrame({
    "Bus": results["bus"][:, BUS_I].astype(int),
    "theta (deg)": results["bus"][:, VA],
    "LMP ($/MWh)": results["bus"][:, LAM_P],
})
bus_df

,Bus,theta (deg),LMP ($/MWh)
0,1,3.2535,16.9774
1,2,-0.7670,26.3845
2,3,-0.4559,30.0000
3,4,0.0000,39.9427
4,5,4.0840,10.0000


`LAM_P` is the multiplier on each bus's power balance, which is the **LMP** in \$/MWh. Bus 4 is the reference bus (angle 0).

### 3. Line flows and shadow prices 

In [8]:
branch_df = pd.DataFrame({
    "Branch": np.arange(1, len(results["branch"]) + 1),
    "From": results["branch"][:, F_BUS].astype(int),
    "To": results["branch"][:, T_BUS].astype(int),
    "Flow (MW)": results["branch"][:, PF],
    "Limit (MW)": results["branch"][:, RATE_A],
})
branch_df["Loading (%)"] = branch_df["Flow (MW)"].abs() / branch_df["Limit (MW)"] * 100

# shadow price of the thermal limit ($/h per MW): one multiplier per line end, at most one is nonzero
branch_df["Shadow price ($/h per MW)"] = results["branch"][:, MU_SF] + results["branch"][:, MU_ST]
branch_df

,Branch,From,To,Flow (MW),Limit (MW),Loading (%),Shadow price ($/h per MW)
0,1,1,2,249.7168,400.0000,62.4292,0.0000
1,2,1,4,186.7884,426.0000,43.8470,0.0000
2,3,1,5,-226.5052,426.0000,53.1702,0.0000
3,4,2,3,-50.2832,426.0000,11.8036,0.0000
4,5,3,4,-26.7884,426.0000,6.2884,0.0000
5,6,4,5,-240.0000,240.0000,100.0000,62.3220


A negative flow means power moves against the listed from → to direction. `MU_SF` and `MU_ST` are the multipliers on the limit at the from and to end of a line. Only one can be nonzero, so adding them gives the line's shadow price.

**Summary:** the solver converges with a total cost of *\$17,479.90/h*. Branch 6 is the only line at its limit (240 MW, flowing from Bus 5 to Bus 4) with a shadow price of *\$62.32/MWh* per MW. LMPs range from *\$10.00* at Bus 5 to *\$39.94* at Bus 4. These values match the Pyomo/HiGHS solution.

## Reference Values for the Pyomo Notebook

The cell below prints the arrays in a form that can be pasted into the Pyomo notebook for the comparison.

In [9]:
pp_lmp = results["bus"][:, LAM_P]
pp_mu = results["branch"][:, MU_SF] + results["branch"][:, MU_ST]
pp_cost = results["f"]

print("pp_lmp =", np.array2string(pp_lmp, precision=8, separator=", "))
print("pp_mu =", np.array2string(pp_mu, precision=8, separator=", "))
print("pp_cost =", pp_cost)

pp_lmp = [16.97735882, 26.38445952, 30.        , 39.94273632, 10.        ]
pp_mu = [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
 62.32204211]
pp_cost = 17479.896925574154
